# StayIQ — Module 4: Dynamic Pricing Suggestion

This notebook demonstrates training a regression model (Gradient Boosting / Random Forest Regressor) to suggest optimized daily room rates. The pricing algorithm dynamically incorporates occupancy trends, date seasonality, competitor pricing benchmarks, and guest cancellation risk scores calculated in Module 1.

### Pipeline Steps:
1. **Load data**: Read historical booking prices and market records.
2. **Feature construction**: Incorporate date features, seasonality indexes, competitor rates, occupancy coefficients, and cancellation risk forecasts.
3. **Train/Test Split**: Divide records into train/test sets.
4. **Model Training**: Fit a Gradient Boosting Regression model.
5. **Evaluation**: Compute Mean Absolute Error (MAE), RMSE, and R2 metrics.
6. **Export Model**: Serialize the trained model to `backend/app/models/` for dynamic pricing endpoint inference.

In [ ]:
# 1. Imports and setup
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [ ]:
# 2. Load or Generate Historical Pricing Data
data_path = "../data/historical_pricing.csv"
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    print("Historical pricing CSV not found. Generating simulated records for training...")
    np.random.seed(42)
    size = 1000
    
    base_prices = np.random.uniform(80.0, 150.0, size=size)
    occupancy_rates = np.random.uniform(0.1, 1.0, size=size)
    competitor_prices = base_prices + np.random.uniform(-15.0, 25.0, size=size)
    seasonality_indices = np.random.choice([0.8, 1.0, 1.2, 1.35], p=[0.2, 0.5, 0.2, 0.1], size=size)
    cancellation_risks = np.random.uniform(0.05, 0.85, size=size)
    
    # Generate target prices using pricing formula with random noise
    target_prices = (
        base_prices 
        + (base_prices * (occupancy_rates - 0.5) * 0.4) 
        + (base_prices * (seasonality_indices - 1.0)) 
        - (base_prices * cancellation_risks * 0.1)
        + np.random.normal(0, 5, size=size)
    )
    
    # Apply competitor bounds alignment
    target_prices = np.clip(target_prices, competitor_prices * 0.85, competitor_prices * 1.15)
    
    df = pd.DataFrame({
        'base_price': base_prices,
        'occupancy_rate': occupancy_rates,
        'competitor_price': competitor_prices,
        'seasonality_index': seasonality_indices,
        'cancellation_risk_score': cancellation_risks,
        'target_price': target_prices
    })

df.head()

In [ ]:
# 3. Split and Train Regression Model
X = df[['base_price', 'occupancy_rate', 'competitor_price', 'seasonality_index', 'cancellation_risk_score']]
y = df['target_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

regressor = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
print("Training Gradient Boosting Regressor...")
regressor.fit(X_train, y_train)
print("Training completed.")

In [ ]:
# 4. Evaluate Pricing Recommendations
y_pred = regressor.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n=== Dynamic Pricing Model Evaluation ===")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"R-squared (R2) Score: {r2:.4f}")

# Plot predicted vs actual
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.6, color='indigo')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Ideal Price ($)')
plt.ylabel('Suggested Recommended Price ($)')
plt.title('Dynamic Pricing: Predicted vs Actual')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
# 5. Export Model for Inference
model_export_path = "../backend/app/models/pricing_model.joblib"
joblib.dump(regressor, model_export_path)
print(f"Pricing regressor model exported successfully to: {model_export_path}")